
# EXPT NO. 3 — Implementation of Convolutional Neural Networks (CNNs) for Image Classification

**Student Name:** Mohamed Aashik S  
**Roll No.:** 24BAD072  

## Aim
To design, implement, and evaluate a Convolutional Neural Network (CNN) for image classification using the CIFAR-10 benchmark image dataset.

## Tasks Covered
- Dataset loading and preparation
- Image preprocessing using normalization
- Training, validation, and testing split
- CNN model construction
- ReLU activation and Max Pooling
- Fully connected and Softmax output layers
- Model compilation using an optimizer and cross-entropy loss
- Model training and evaluation
- Accuracy and loss plots
- Confusion matrix
- Sample predictions
- Misclassified image visualization


## Step 1 — Install / Import Required Libraries

In [ ]:

# If running in Google Colab, TensorFlow and most libraries are already available.
# Uncomment the following line only if a package is missing.

# !pip install -q tensorflow scikit-learn matplotlib seaborn

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print("TensorFlow version:", tf.__version__)


## Step 2 — Load the CIFAR-10 Dataset

In [ ]:

# CIFAR-10 contains 60,000 colour images belonging to 10 classes.
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Convert labels from shape (n, 1) to (n,)
y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Training images:", x_train_full.shape)
print("Training labels:", y_train_full.shape)
print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)


## Step 3 — Visualize Sample Images

In [ ]:

plt.figure(figsize=(10, 6))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_train_full[i])
    plt.title(class_names[y_train_full[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()


## Step 4 — Preprocess and Split the Dataset

In [ ]:

# Normalize pixel values from [0, 255] to [0, 1].
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Use 5,000 images from the training set for validation.
validation_size = 5000

x_val = x_train_full[-validation_size:]
y_val = y_train_full[-validation_size:]

x_train = x_train_full[:-validation_size]
y_train = y_train_full[:-validation_size]

print("Training set:", x_train.shape, y_train.shape)
print("Validation set:", x_val.shape, y_val.shape)
print("Testing set:", x_test.shape, y_test.shape)


## Step 5 — Convert Labels to Categorical Format

In [ ]:

# Convert class labels into one-hot encoded vectors.
y_train_cat = to_categorical(y_train, 10)
y_val_cat = to_categorical(y_val, 10)
y_test_cat = to_categorical(y_test, 10)

print("Example original label:", y_train[0])
print("Example one-hot label:", y_train_cat[0])


## Step 6 — Build the CNN Model

In [ ]:

model = Sequential([
    # First convolution block
    Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),

    # Second convolution block
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    # Third convolution block
    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    # Classification layers
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(10, activation="softmax")
])

model.summary()


## Step 7 — Compile the CNN

In [ ]:

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")


## Step 8 — Train the CNN Model

In [ ]:

# Train the model.
# 15 epochs provides a reasonable lab demonstration while keeping runtime manageable.
history = model.fit(
    x_train,
    y_train_cat,
    epochs=15,
    batch_size=64,
    validation_data=(x_val, y_val_cat),
    verbose=1
)


## Step 9 — Record Training and Validation Performance

In [ ]:

# Get the final recorded values.
final_train_accuracy = history.history["accuracy"][-1]
final_val_accuracy = history.history["val_accuracy"][-1]
final_train_loss = history.history["loss"][-1]
final_val_loss = history.history["val_loss"][-1]

print(f"Final Training Accuracy   : {final_train_accuracy:.4f}")
print(f"Final Validation Accuracy : {final_val_accuracy:.4f}")
print(f"Final Training Loss       : {final_train_loss:.4f}")
print(f"Final Validation Loss     : {final_val_loss:.4f}")


## Step 10 — Evaluate the Model on the Test Set

In [ ]:

test_loss, test_accuracy = model.evaluate(
    x_test,
    y_test_cat,
    verbose=0
)

print(f"Testing Loss    : {test_loss:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")


## Step 11 — Plot Accuracy vs Epoch

In [ ]:

plt.figure(figsize=(8, 5))

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.title("Accuracy vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()


## Step 12 — Plot Loss vs Epoch

In [ ]:

plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Loss vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


## Step 13 — Generate Predictions

In [ ]:

# Predict class probabilities for the test images.
pred_probabilities = model.predict(x_test, verbose=0)
y_pred = np.argmax(pred_probabilities, axis=1)

print("Predictions generated successfully.")
print("First 10 predicted classes:")
print([class_names[i] for i in y_pred[:10]])


## Step 14 — Confusion Matrix

In [ ]:

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## Step 15 — Classification Report

In [ ]:

print(classification_report(
    y_test,
    y_pred,
    target_names=class_names,
    digits=4
))


## Step 16 — Display Sample Predictions

In [ ]:

plt.figure(figsize=(12, 8))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_test[i])

    true_label = class_names[y_test[i]]
    predicted_label = class_names[y_pred[i]]

    title = f"True: {true_label}\nPred: {predicted_label}"
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()


## Step 17 — Display Misclassified Images

In [ ]:

# Find images where prediction does not match the true class.
misclassified_indices = np.where(y_pred != y_test)[0]

print("Number of misclassified images:", len(misclassified_indices))

plt.figure(figsize=(12, 8))

num_to_show = min(12, len(misclassified_indices))

for i in range(num_to_show):
    idx = misclassified_indices[i]

    plt.subplot(3, 4, i + 1)
    plt.imshow(x_test[idx])

    true_label = class_names[y_test[idx]]
    predicted_label = class_names[y_pred[idx]]

    plt.title(f"True: {true_label}\nPred: {predicted_label}")
    plt.axis("off")

plt.tight_layout()
plt.show()


## Step 18 — Automatically Generate a Performance Summary

In [ ]:

print("=" * 55)
print("CNN PERFORMANCE SUMMARY")
print("=" * 55)
print(f"Training Accuracy   : {final_train_accuracy:.2%}")
print(f"Validation Accuracy : {final_val_accuracy:.2%}")
print(f"Training Loss       : {final_train_loss:.4f}")
print(f"Validation Loss     : {final_val_loss:.4f}")
print(f"Testing Accuracy    : {test_accuracy:.2%}")
print(f"Testing Loss        : {test_loss:.4f}")
print(f"Misclassified       : {len(misclassified_indices)} / {len(y_test)}")
print("=" * 55)



## Step 19 — Strengths and Limitations

### Strengths
CNNs automatically learn useful visual features directly from images and preserve spatial relationships through convolution operations. They use shared weights, making them more parameter-efficient than fully connected networks for image data. CNNs can achieve strong performance on image classification tasks.

### Limitations
CNNs can require significant computational resources and training time. Their performance depends on suitable preprocessing, architecture design, and hyperparameter selection. They may also overfit when the training dataset is limited or when the model is unnecessarily complex.



## Step 20 — Final Lab Observations

The CNN was successfully implemented using the CIFAR-10 dataset. The images were normalized and divided into training, validation, and testing sets. The model used convolutional layers with ReLU activation, Max Pooling, fully connected layers, and a Softmax output layer. Training and validation accuracy/loss were visualized using graphs, while the confusion matrix and sample predictions were used to analyze classification performance. The final testing accuracy displayed above should be reported in the lab record.
